# EduAnalytics — Exploratory Data Analysis

**Dataset:** UCI Student Performance (student-mat.csv)  
**Target:** `G3` — Final exam grade (0–20)

This notebook covers:
1. Data loading & basic inspection
2. Descriptive statistics
3. Target distribution
4. Grade progression (G1 → G2 → G3)
5. Categorical feature analysis
6. Numeric feature distributions
7. Correlation heatmap
8. Key insights summary

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DATA_PATH = os.path.join('..', 'data', 'student-mat.csv')
print('EduAnalytics EDA — ready')

## 1. Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH, sep=';')
print(f'Shape: {df.shape}')
df.head()

## 2. Basic Inspection

In [ ]:
print('Data Types:')
print(df.dtypes)
print(f'\nMissing Values: {df.isnull().sum().sum()}')

In [ ]:
df.describe().round(2)

## 3. Target Variable — G3 Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['G3'], bins=21, kde=True, color='steelblue', ax=axes[0])
axes[0].axvline(df['G3'].mean(), color='red', linestyle='--', label=f'Mean: {df["G3"].mean():.2f}')
axes[0].axvline(df['G3'].median(), color='orange', linestyle='--', label=f'Median: {df["G3"].median():.2f}')
axes[0].set_title('G3 Score Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Final Grade (G3)')
axes[0].legend()

sns.boxplot(y=df['G3'], color='lightcoral', ax=axes[1])
axes[1].set_title('G3 Box Plot', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Mean={df["G3"].mean():.2f}  Median={df["G3"].median():.2f}  Std={df["G3"].std():.2f}')

## 4. Grade Progression G1 → G2 → G3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, color in zip(axes, ['G1','G2','G3'], ['#4C72B0','#DD8452','#55A868']):
    sns.histplot(df[col], bins=21, kde=True, color=color, ax=ax)
    ax.set_title(f'{col}  (mean={df[col].mean():.1f})', fontweight='bold')
plt.suptitle('Grade Progression Across Three Periods', fontsize=14)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(df[['G1','G2','G3']].corr(), annot=True, fmt='.3f', cmap='coolwarm', ax=ax, square=True)
ax.set_title('Grade Correlation', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Categorical Feature Analysis

In [ ]:
cat_cols = ['sex','address','famsize','Pstatus','schoolsup','famsup','higher','internet','romantic']
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    order = df.groupby(col)['G3'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='G3', order=order, palette='Set2', ax=axes[i])
    axes[i].set_title(f'G3 by {col}', fontweight='bold')
    axes[i].set_xlabel('')
plt.suptitle('G3 by Categorical Features', fontsize=15)
plt.tight_layout()
plt.show()

## 6. Numeric Feature Distributions

In [ ]:
num_cols = ['age','Medu','Fedu','traveltime','studytime','failures','absences','famrel','freetime','goout','Dalc','Walc','health']
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col], bins=10, color='#4C72B0', ax=axes[i])
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Numeric Feature Distributions', fontsize=15)
plt.tight_layout()
plt.show()

## 7. Full Correlation Heatmap

In [ ]:
df_enc = df.copy()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

corr = df_enc.corr()
fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, linewidths=0.3, ax=ax, annot_kws={'size':7}, square=True)
ax.set_title('Full Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
g3_corr = corr['G3'].drop('G3').sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in g3_corr.values]
ax.barh(g3_corr.index, g3_corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with G3', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()
print('Top correlations with G3:')
print(g3_corr.head(5))

## 8. Key Insights

In [ ]:
print('=' * 50)
print('  EduAnalytics — Key Insights')
print('=' * 50)
print(f'  Total students        : {len(df)}')
print(f'  Average G3            : {df["G3"].mean():.2f} / 20')
print(f'  Students with G3 = 0  : {(df["G3"]==0).sum()} ({(df["G3"]==0).mean()*100:.1f}%)')
print(f'  Students with G3 >= 15: {(df["G3"]>=15).sum()} ({(df["G3"]>=15).mean()*100:.1f}%)')
print(f'  Top predictor         : {g3_corr.index[0]} (r={g3_corr.iloc[0]:.3f})')
print(f'  Strongest negative    : {g3_corr.index[-1]} (r={g3_corr.iloc[-1]:.3f})')
print('=' * 50)